In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "functions").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "functions").exists() and (parent / "data").exists():
            REPO_ROOT = parent
            break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pickle

import numpy as np
import pandas as pd

from experiments.real.window_processing import prepare_frechet_data
from experiments.real.ai_permutation import CANONICAL_PREDICTOR_RUNS, permutation_R2

SEED = 20251225

Using R installation at: C:\Program Files\R\R-4.4.1


In [2]:
features_aireadi = pd.read_csv('data/aireadi_window120.csv')
feature_names = ['Mean', 'CV', 'MAD']

# Processing for Frechet regression with selected features
aireadi_processed = prepare_frechet_data(features_aireadi, feature_names=feature_names)

Processed 964 patients with sufficient data
Average windows per patient: 458.69
Calculating latent correlations... Done!


In [3]:
# Load metadata that includes predictors
metadata = pd.read_csv(os.path.join('data', 'ai_readi_metadata_cleaned.csv'))
metadata.head()

,id,age,study_group,LDL-C,HDL-C,Total-C,TG,HbA1c,log(HbA1c),log(TG)
0,1001,69,pre_diabetes_lifestyle_controlled,94.341690,92.0,201.0,83.0,5.7,1.740466,4.418841
1,1002,69,healthy,133.485054,41.0,202.0,153.0,5.6,1.722767,5.030438
2,1003,82,oral_medication_and_or_non_insulin_injectable_...,105.112772,63.0,192.0,137.0,7.8,2.054124,4.919981
3,1004,61,oral_medication_and_or_non_insulin_injectable_...,59.674544,54.0,128.0,70.0,7.2,1.974081,4.248495
4,1005,58,insulin_dependent,74.956702,52.0,141.0,70.0,9.5,2.251292,4.248495


In [4]:
predictor_list = ['HbA1c', 'log(TG)', 'HDL-C', 'Total-C']

### Make permutation function

In [4]:
CANONICAL_RESULTS_DIR = REPO_ROOT / 'results' / 'real' / 'R2'
CANONICAL_TABLE_PATH = REPO_ROOT / 'results' / 'tables' / 'R2_adjusted_pvalues.tex'
CANONICAL_TABLE_ORDER = [
    'HbA1c',
    'log(TG)',
    'HDL-C',
    'Total-C',
    'HbA1c_log(TG)_HDL-C_Total-C',
]
DISPLAY_NAME_MAP = {
    'HbA1c': 'HbA1c',
    'log(TG)': 'TG',
    'HDL-C': 'HDL-C',
    'Total-C': 'Total-C',
    'HbA1c_log(TG)_HDL-C_Total-C': 'All predictors',
}

def predictor_key(predictor_list):
    if isinstance(predictor_list, str):
        predictor_list = [predictor_list]
    return '_'.join(predictor_list)

`ai_permutation.py` script itself runs the following cell. This is described here for completeness

In [ ]:
# Run the permutation tests.
# This should be run in HPC cluster, or takes 5~6 hours in a standard laptop
for predictor_list in CANONICAL_PREDICTOR_RUNS:
    all_res = permutation_R2(
        predictor_list,
        n_permutations=2000,
        seed=SEED,
        njobs=10,
        aireadi_processed=aireadi_processed,
        metadata=metadata,
    )

To run a model with a single predictor, uncomment the following cell and run

In [ ]:
# Approx 50 min with 10 cores in a standard laptop
# all_res = permutation_R2(
#     ['HbA1c'],
#     n_permutations=2000,
#     seed=SEED,
#     njobs=10,
#     aireadi_processed=aireadi_processed,
#     metadata=metadata,
# )

### Load permutation results and adjust p-values via Westfall--Young min-p single-step procedure


In [7]:
# Westfall-Young min-p procedure
def minP_adjust(table: np.ndarray, origin: np.ndarray, stepdown: bool = False):
    """
    table: (B, m) permutation statistics (here, R^2), same permutation applied across components
    origin: (m,) observed statistics
    Returns dict with:
      - p_unadj: unadjusted permutation p-values
      - p_minP: single-step min-P adjusted p-values (FWER control)
      - p_minP_step: step-down min-P adjusted p-values (FWER control)
    """
    table = np.asarray(table, dtype=float)
    origin = np.asarray(origin, dtype=float)
    assert table.ndim == 2, 'table must be 2D (B x m)'
    B, _ = table.shape

    ge_obs = (table >= origin).sum(axis=0)
    p_unadj = (1.0 + ge_obs) / (B + 1.0)

    P_perm = np.empty_like(table, dtype=float)
    for j in range(table.shape[1]):
        col = table[:, j]
        s = np.sort(col)
        first_idx = np.searchsorted(s, col, side='left')
        ge_counts = B - first_idx
        P_perm[:, j] = (1.0 + ge_counts) / (B + 1.0)

    Q_all = P_perm.min(axis=1)
    p_minP = (1.0 + (Q_all[:, None] <= p_unadj[None, :]).sum(axis=0)) / (B + 1.0)

    if not stepdown:
        return {'p_unadj': p_unadj, 'p_minP': p_minP, 'p_minP_step': None}

    order = np.argsort(p_unadj)
    p_unadj_sorted = p_unadj[order]
    P_perm_sorted = P_perm[:, order]
    cummin_right = np.minimum.accumulate(P_perm_sorted[:, ::-1], axis=1)[:, ::-1]

    p_minP_step_sorted = np.empty(len(order), dtype=float)
    prev = 0.0
    for k in range(len(order)):
        Qk = cummin_right[:, k]
        p_raw = (1.0 + (Qk <= p_unadj_sorted[k]).sum()) / (B + 1.0)
        prev = max(prev, p_raw)
        p_minP_step_sorted[k] = prev

    inv = np.empty_like(order)
    inv[order] = np.arange(len(order))
    p_minP_step = p_minP_step_sorted[inv]

    return {'p_unadj': p_unadj, 'p_minP': p_minP, 'p_minP_step': p_minP_step}


In [8]:
# Read the canonical results, update add-one p-values and Westfall-Young adjustments,
# and write them back to results/real/R2.
updated_keys = []

for predictor_list in CANONICAL_PREDICTOR_RUNS:
    key = predictor_key(predictor_list)
    file_path = CANONICAL_RESULTS_DIR / f'permutation_R2_{key}.pkl'
    with open(file_path, 'rb') as f:
        all_results = pickle.load(f)

    p_values = pd.DataFrame(columns=all_results['original_R2'].columns)
    for feature in all_results['original_R2'].columns:
        orig_value = all_results['original_R2'][feature].values[0]
        perm_values = all_results['permuted_R2s'][feature].values
        p_values.loc[0, feature] = (np.sum(perm_values >= orig_value) + 1) / (len(perm_values) + 1)

    table = all_results['permuted_R2s'].values[:, :-1]
    origin = all_results['original_R2'].values[:, :-1]
    minp_results = minP_adjust(table, origin, stepdown=False)

    all_results['p_values'] = p_values
    all_results['p_adjusted'] = minp_results['p_minP']

    with open(file_path, 'wb') as f:
        pickle.dump(all_results, f)

    updated_keys.append(key)

print('Updated canonical results:')
for key in updated_keys:
    print(f'  - {key}')

Updated canonical results:
  - HbA1c
  - log(TG)
  - HDL-C
  - Total-C
  - HbA1c_log(TG)_HDL-C_Total-C


# Table of $R^2$ values
Construct the canonical paper table only


In [9]:
# Load the canonical paper-relevant results only.
results_summary = {}

for predictor_list in CANONICAL_PREDICTOR_RUNS:
    key = predictor_key(predictor_list)
    file_path = CANONICAL_RESULTS_DIR / f'permutation_R2_{key}.pkl'
    with open(file_path, 'rb') as f:
        all_results = pickle.load(f)

    r2_values = all_results['original_R2'].copy()
    if 'total' in r2_values.columns:
        r2_values = r2_values.drop(columns=['total'])

    results_summary[key] = {
        'R2': r2_values.iloc[0].to_dict(),
        'p_adjusted': all_results['p_adjusted'],
    }

print('Canonical results loaded:')
for key in results_summary.keys():
    print(f'  - {key}')

Canonical results loaded:
  - HbA1c
  - log(TG)
  - HDL-C
  - Total-C
  - HbA1c_log(TG)_HDL-C_Total-C


In [14]:
# Create the final canonical table from the explicit paper-relevant model set.
final_results = []

for predictor_key_name in CANONICAL_TABLE_ORDER:
    data = results_summary[predictor_key_name]
    row = {'Predictor': DISPLAY_NAME_MAP[predictor_key_name]}

    r2_dict = data['R2']
    p_adjusted = data['p_adjusted']
    for i, feature_name in enumerate(r2_dict.keys()):
        col_name = 'lcor' if feature_name == 'latentcor' else feature_name
        r2_value = r2_dict[feature_name]
        p_value = p_adjusted[i]
        p_str = '$<0.001$' if p_value < 0.001 else f'{p_value:.3f}'
        row[col_name] = f'{r2_value:.3f} ({p_str})'

    final_results.append(row)

results_df = pd.DataFrame(final_results)
results_df = results_df[['Predictor', 'Mean', 'CV', 'MAD', 'lcor']]

print("Final Results Table:")
print("=" * 80)
display(results_df)

Final Results Table:


,Predictor,Mean,CV,MAD,lcor
0,HbA1c,0.557 ($<0.001$),0.005 (0.047),0.060 ($<0.001$),0.072 ($<0.001$)
1,TG,0.043 ($<0.001$),0.014 ($<0.001$),0.003 (0.288),0.009 (0.004)
2,HDL-C,0.068 ($<0.001$),0.014 (0.002),0.010 (0.004),0.040 ($<0.001$)
3,Total-C,0.036 ($<0.001$),0.004 (0.138),0.005 (0.085),0.022 ($<0.001$)
4,All predictors,0.575 ($<0.001$),0.030 ($<0.001$),0.079 ($<0.001$),0.114 ($<0.001$)


In [15]:
# Export to LaTeX booktabs format.
os.makedirs(CANONICAL_TABLE_PATH.parent, exist_ok=True)

latex_str = results_df.to_latex(
    index=False,
    escape=False,
    column_format='l' + 'c' * (len(results_df.columns) - 1),
    caption='$R^2$ values with adjusted $p$-values for different predictor combinations',
    label='tab:r2_results',
    position='htbp'
)

lines = latex_str.split('\n')
booktabs_lines = []
hline_count = 0

for line in lines:
    if '\\hline' in line:
        hline_count += 1
        if hline_count == 1:
            booktabs_lines.append(line.replace('\\hline', '\\toprule'))
        elif hline_count == 2:
            booktabs_lines.append(line.replace('\\hline', '\\midrule'))
        else:
            booktabs_lines.append(line.replace('\\hline', '\\bottomrule'))
    elif '\\begin{table}' in line:
        booktabs_lines.append(line)
        booktabs_lines.append('\\centering')
    else:
        booktabs_lines.append(line)

latex_str = '\n'.join(booktabs_lines)
with open(CANONICAL_TABLE_PATH, 'w') as f:
    f.write(latex_str)

print(f"LaTeX table saved to: {CANONICAL_TABLE_PATH}")


LaTeX table saved to: G:\My Drive\Experiments\NonparaFrechet\results\tables\R2_adjusted_pvalues.tex
